# Image Properties Analysis
**Dataset:** 800 images (400 urban, 400 landscape) scored across 15 architectural/aesthetic properties using Claude API.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('all_images_15_properties.csv')

# Define properties and color palette
properties = [
    'levels_of_scale', 'strong_centers', 'boundaries', 'alternating_repetition',
    'positive_space', 'good_shape', 'local_symmetries', 'deep_interlock_and_ambiguity',
    'contrast', 'gradients', 'roughness', 'echoes', 'the_void',
    'simplicity_and_inner_calm', 'not_separateness'
]

pretty_labels = [
    'Levels of Scale', 'Strong Centers', 'Boundaries', 'Alternating Repetition',
    'Positive Space', 'Good Shape', 'Local Symmetries', 'Deep Interlock & Ambiguity',
    'Contrast', 'Gradients', 'Roughness', 'Echoes', 'The Void',
    'Simplicity & Inner Calm', 'Not-Separateness'
]

COLOR_URBAN = '#E74C3C'
COLOR_LANDSCAPE = '#2980B9'
COLOR_PALETTE = {'urban': COLOR_URBAN, 'landscape': COLOR_LANDSCAPE}

df_urban = df[df['image_type'] == 'urban']
df_landscape = df[df['image_type'] == 'landscape']

print(f"Dataset shape: {df.shape}")
print(f"Image types: {df['image_type'].value_counts().to_dict()}")
print(f"Score range: {df[properties].min().min()} – {df[properties].max().max()}")
df.head()

## 2. Descriptive Statistics by Image Type

In [ ]:
print("=== URBAN ===")
display(df_urban[properties].describe().round(2))

print("\n=== LANDSCAPE ===")
display(df_landscape[properties].describe().round(2))

## 3. Mean Scores per Property — Urban vs Landscape (Bar Chart)

In [ ]:
means_urban = df_urban[properties].mean()
means_landscape = df_landscape[properties].mean()

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(properties))
width = 0.38

bars1 = ax.bar(x - width/2, means_urban, width, label='Urban', color=COLOR_URBAN, alpha=0.88, edgecolor='white')
bars2 = ax.bar(x + width/2, means_landscape, width, label='Landscape', color=COLOR_LANDSCAPE, alpha=0.88, edgecolor='white')

ax.set_xlabel('Property', fontsize=12)
ax.set_ylabel('Mean Score (1–10)', fontsize=12)
ax.set_title('Mean Property Scores: Urban vs Landscape', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(pretty_labels, rotation=45, ha='right', fontsize=9)
ax.set_ylim(0, 10.5)
ax.legend(fontsize=11)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

# Annotate difference
for i in range(len(properties)):
    diff = means_urban.iloc[i] - means_landscape.iloc[i]
    color = 'green' if diff > 0 else 'darkred'
    ax.text(x[i], max(means_urban.iloc[i], means_landscape.iloc[i]) + 0.25,
            f'{diff:+.1f}', ha='center', va='bottom', fontsize=7, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('01_mean_scores_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Radar / Spider Chart — Profile Comparison

In [ ]:
urban_vals = df_urban[properties].mean().tolist()
landscape_vals = df_landscape[properties].mean().tolist()

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=urban_vals + [urban_vals[0]],
    theta=pretty_labels + [pretty_labels[0]],
    fill='toself',
    name='Urban',
    line_color=COLOR_URBAN,
    fillcolor='rgba(231,76,60,0.25)'
))

fig.add_trace(go.Scatterpolar(
    r=landscape_vals + [landscape_vals[0]],
    theta=pretty_labels + [pretty_labels[0]],
    fill='toself',
    name='Landscape',
    line_color=COLOR_LANDSCAPE,
    fillcolor='rgba(41,128,185,0.25)'
))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
    showlegend=True,
    title=dict(text='Mean Property Profile: Urban vs Landscape', x=0.5, font_size=16),
    width=700, height=600
)

fig.show()
fig.write_html('02_radar_chart.html')

## 5. Score Distribution — Violin + Box Plots per Property

In [ ]:
df_melted = df.melt(id_vars=['image_name', 'image_type'], value_vars=properties,
                    var_name='property', value_name='score')
df_melted['property_label'] = df_melted['property'].map(dict(zip(properties, pretty_labels)))

fig = px.violin(
    df_melted,
    x='property_label',
    y='score',
    color='image_type',
    box=True,
    points=False,
    color_discrete_map={'urban': COLOR_URBAN, 'landscape': COLOR_LANDSCAPE},
    title='Score Distribution per Property by Image Type',
    labels={'property_label': 'Property', 'score': 'Score', 'image_type': 'Type'}
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=1100, height=550,
    violingap=0.2,
    violingroupgap=0.1
)

fig.show()
fig.write_html('03_violin_distributions.html')

## 6. Correlation Heatmaps (Urban & Landscape side by side)

In [ ]:
corr_urban = df_urban[properties].corr()
corr_landscape = df_landscape[properties].corr()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, corr, title, cmap in zip(
    axes,
    [corr_urban, corr_landscape],
    ['Urban — Property Correlations', 'Landscape — Property Correlations'],
    ['RdYlGn', 'RdYlBu']
):
    im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(properties)))
    ax.set_yticks(range(len(properties)))
    ax.set_xticklabels(pretty_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(pretty_labels, fontsize=8)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Annotate cells
    for i in range(len(properties)):
        for j in range(len(properties)):
            val = corr.iloc[i, j]
            text_color = 'black' if abs(val) < 0.6 else 'white'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=6.5, color=text_color)

plt.tight_layout()
plt.savefig('04_correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Score Difference Heatmap (Urban − Landscape)

In [ ]:
diff = df_urban[properties].mean() - df_landscape[properties].mean()

fig = go.Figure(go.Heatmap(
    z=[diff.values],
    x=pretty_labels,
    y=['Urban − Landscape'],
    colorscale='RdBu_r',
    zmid=0,
    text=[[f'{v:+.2f}' for v in diff.values]],
    texttemplate='%{text}',
    textfont_size=11,
    showscale=True,
    colorbar_title='Δ Score'
))

fig.update_layout(
    title=dict(text='Mean Score Difference: Urban − Landscape', x=0.5, font_size=16),
    xaxis_tickangle=-40,
    height=280,
    width=1050,
    margin=dict(b=120)
)

fig.show()
fig.write_html('05_difference_heatmap.html')

## 8. Score Frequency Distribution (Histograms per Property)

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

bins = np.arange(1, 12) - 0.5  # bins from 0.5 to 10.5

for i, (prop, label) in enumerate(zip(properties, pretty_labels)):
    ax = axes[i]
    u_vals = df_urban[prop]
    l_vals = df_landscape[prop]

    ax.hist(u_vals, bins=bins, alpha=0.65, color=COLOR_URBAN, label='Urban', edgecolor='white', linewidth=0.5)
    ax.hist(l_vals, bins=bins, alpha=0.65, color=COLOR_LANDSCAPE, label='Landscape', edgecolor='white', linewidth=0.5)

    ax.axvline(u_vals.mean(), color=COLOR_URBAN, linestyle='--', linewidth=1.5, alpha=0.9)
    ax.axvline(l_vals.mean(), color=COLOR_LANDSCAPE, linestyle='--', linewidth=1.5, alpha=0.9)

    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xlabel('Score', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.set_xlim(0.5, 10.5)
    ax.tick_params(labelsize=7)

patch_u = mpatches.Patch(color=COLOR_URBAN, alpha=0.7, label='Urban')
patch_l = mpatches.Patch(color=COLOR_LANDSCAPE, alpha=0.7, label='Landscape')
fig.legend(handles=[patch_u, patch_l], loc='upper right', fontsize=11, framealpha=0.9)

fig.suptitle('Score Frequency Distributions per Property', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('06_histograms_per_property.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Per-Image Total Score Distribution

In [ ]:
df['total_score'] = df[properties].sum(axis=1)
df['mean_score'] = df[properties].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, ylabel in zip(axes, ['total_score', 'mean_score'], ['Total Score', 'Mean Score']):
    for itype, color in COLOR_PALETTE.items():
        sub = df[df['image_type'] == itype][col]
        ax.hist(sub, bins=25, alpha=0.7, color=color, label=itype.capitalize(), edgecolor='white')
        ax.axvline(sub.mean(), color=color, linestyle='--', linewidth=2)
    ax.set_xlabel(ylabel, fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'Distribution of {ylabel} per Image', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('07_total_mean_score_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print("Mean total scores:")
print(df.groupby('image_type')['total_score'].describe().round(2))

## 10. Scatter Matrix (Key Properties)

In [ ]:
key_props = ['levels_of_scale', 'contrast', 'roughness', 'the_void', 'not_separateness']
key_labels = ['Levels of Scale', 'Contrast', 'Roughness', 'The Void', 'Not-Separateness']

fig = px.scatter_matrix(
    df,
    dimensions=key_props,
    color='image_type',
    color_discrete_map={'urban': COLOR_URBAN, 'landscape': COLOR_LANDSCAPE},
    labels=dict(zip(key_props, key_labels)),
    title='Scatter Matrix of Key Properties',
    opacity=0.4
)

fig.update_traces(marker_size=3, diagonal_visible=False)
fig.update_layout(width=850, height=800)
fig.show()
fig.write_html('08_scatter_matrix.html')

## 11. Ranked Property Importance (Mean + Std)

In [ ]:
stats = df.groupby('image_type')[properties].agg(['mean', 'std']).round(3)

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)

for ax, itype, color in zip(axes, ['urban', 'landscape'], [COLOR_URBAN, COLOR_LANDSCAPE]):
    means = stats.loc[itype, (slice(None), 'mean')].values
    stds  = stats.loc[itype, (slice(None), 'std')].values
    order = np.argsort(means)[::-1]

    ax.barh(
        [pretty_labels[i] for i in order],
        means[order],
        xerr=stds[order],
        color=color, alpha=0.82,
        ecolor='gray', capsize=4, edgecolor='white'
    )
    ax.set_xlim(0, 11)
    ax.set_xlabel('Mean Score ± Std', fontsize=11)
    ax.set_title(f'{itype.capitalize()} — Ranked Properties', fontsize=13, fontweight='bold')
    ax.xaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('09_ranked_properties.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Interactive Parallel Coordinates Plot

In [ ]:
df_sample = df.groupby('image_type').apply(lambda x: x.sample(min(150, len(x)), random_state=42)).reset_index(drop=True)
df_sample['type_code'] = (df_sample['image_type'] == 'urban').astype(int)

dimensions = [
    dict(range=[1, 10], label=pretty_labels[i], values=df_sample[prop])
    for i, prop in enumerate(properties)
]

fig = go.Figure(go.Parcoords(
    line=dict(
        color=df_sample['type_code'],
        colorscale=[[0, COLOR_LANDSCAPE], [1, COLOR_URBAN]],
        showscale=True,
        cmin=0, cmax=1,
        colorbar=dict(
            tickvals=[0, 1],
            ticktext=['Landscape', 'Urban'],
            title='Type'
        )
    ),
    dimensions=dimensions
))

fig.update_layout(
    title=dict(text='Parallel Coordinates — All 15 Properties (sampled)', x=0.5, font_size=15),
    width=1150, height=520
)

fig.show()
fig.write_html('10_parallel_coordinates.html')

## 13. Statistical Summary Table

In [ ]:
from scipy import stats as scipy_stats

summary_rows = []
for prop, label in zip(properties, pretty_labels):
    u = df_urban[prop]
    l = df_landscape[prop]
    t_stat, p_val = scipy_stats.ttest_ind(u, l)
    summary_rows.append({
        'Property': label,
        'Urban Mean': round(u.mean(), 2),
        'Urban Std':  round(u.std(), 2),
        'Landscape Mean': round(l.mean(), 2),
        'Landscape Std':  round(l.std(), 2),
        'Difference': round(u.mean() - l.mean(), 2),
        't-stat': round(t_stat, 3),
        'p-value': round(p_val, 4),
        'Significant (p<0.05)': 'Yes' if p_val < 0.05 else 'No'
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Difference', ascending=False)
display(summary_df.style.background_gradient(subset=['Difference'], cmap='RdBu_r')
        .highlight_between(subset=['Significant (p<0.05)'], axis=None))